<a href="https://colab.research.google.com/github/suhelm/GEN_AI/blob/main/RAG/SimpleRAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q chromadb sentence-transformers transformers datasets torch

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 38.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 76.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 58.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.3/206.3 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 2.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently t

In [1]:
from datasets import load_dataset

passages = load_dataset("rag-datasets/rag-mini-wikipedia", "text-corpus", split="passages")
qa = load_dataset("rag-datasets/rag-mini-wikipedia", "question-answer", split="test")

print(len(passages), len(qa))
print(passages[0]["passage"][:200])
print(qa[0])

README.md:   0%|          | 0.00/719 [00:00<?, ?B/s]

data/passages.parquet/part.0.parquet: reconstructing file:   0%|          |  0.00B /  797kB            

data/passages.parquet/part.0.parquet: downloading bytes:           |  0.00B            

Generating passages split:   0%|          | 0/3200 [00:00<?, ? examples/s]

data/test.parquet/part.0.parquet: reconstructing file:   0%|          |  0.00B / 54.4kB            

data/test.parquet/part.0.parquet: downloading bytes:           |  0.00B            

Generating test split:   0%|          | 0/918 [00:00<?, ? examples/s]

3200 918
Uruguay (official full name in  ; pron.  , Eastern Republic of  Uruguay) is a country located in the southeastern part of South America.  It is home to 3.3 million people, of which 1.7 million live in
{'question': 'Was Abraham Lincoln the sixteenth President of the United States?', 'answer': 'yes', 'id': 0}


**Step 3: Build the vector store with ChromaDB. ** This is the indexing phase. Chroma calls the embedding model on each passage and stores the text and its vector together. PersistentClient saves to disk, so the if check skips re-indexing on later runs. It takes about a minute on CPU.

In [2]:
import chromadb
from chromadb.utils import embedding_functions

client = chromadb.PersistentClient(path="./chroma_db")

embed_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)

collection = client.get_or_create_collection(
    name="wiki_rag",
    embedding_function=embed_fn
)

if collection.count() == 0:
    docs = passages["passage"]
    ids = [str(i) for i in range(len(docs))]
    batch = 500
    for i in range(0, len(docs), batch):
        collection.add(documents=docs[i:i+batch], ids=ids[i:i+batch])

print("Stored passages:", collection.count())

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Stored passages: 3200


Step 4: Load the free Hugging Face LLM

In [5]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/flan-t5-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

def generator(prompt, **kwargs):
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
    with torch.no_grad():
        output = model.generate(**inputs, max_new_tokens=100)
    text = tokenizer.decode(output[0], skip_special_tokens=True)
    return [{"generated_text": text}]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  990MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Step 5: Write the retrieve and generate functions

In [6]:
def retrieve(question, k=3):
    res = collection.query(query_texts=[question], n_results=k)
    return res["documents"][0]

def rag_answer(question, k=3):
    chunks = retrieve(question, k)
    context = "\n\n".join(chunks)

    prompt = (
        "Answer the question using only the context below. "
        "If the answer is not in the context, say \"I don't know\".\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {question}\nAnswer:"
    )
    answer = generator(prompt, truncation=True)[0]["generated_text"]
    return answer, chunks

Step 6: Test it

In [7]:
for i in range(3):
    q = qa[i]["question"]
    answer, chunks = rag_answer(q)
    print("Q:           ", q)
    print("RAG answer:  ", answer)
    print("Ground truth:", qa[i]["answer"])
    print("Top chunk:   ", chunks[0][:150], "...")
    print("-" * 60)

Q:            Was Abraham Lincoln the sixteenth President of the United States?
RAG answer:   yes
Ground truth: yes
Top chunk:    Young Abraham Lincoln ...
------------------------------------------------------------
Q:            Did Lincoln sign the National Banking Act of 1863?
RAG answer:   Yes
Ground truth: yes
Top chunk:    Lincoln believed in the Whig theory of the presidency, which left Congress to write the laws while he signed them, vetoing only those bills that threa ...
------------------------------------------------------------
Q:            Did his mother die of pneumonia?
RAG answer:   I don't know
Ground truth: no
Top chunk:    An autopsy performed after his death revealed grave problems with his stomach and other organs of his abdomen, along with damage to his brain. Despite ...
------------------------------------------------------------


Step 7: See what RAG adds

In [9]:
q = qa[0]["question"]
no_rag = generator(f"Question: {q}\nAnswer:")[0]["generated_text"]
with_rag, _ = rag_answer(q)

print("Without RAG:", no_rag)
print("With RAG:   ", with_rag)

Without RAG: yes
With RAG:    yes
